# Z3-C#-16d — Convergence à l'échelle : l'encodage décide de la tractabilité

*Un planificateur de menus tient sur un corpus jouet — tiendra-t-il sur le corpus réel ?
À cette échelle, la question n'est plus « le solveur est-il assez puissant ? » mais
« le problème est-il écrit dans une forme que le solveur peut traverser ? »*

Jumeau C# natif du grain [16d Python](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb),
lui-même port fidèle du carnet `09_Meal_Planner_Convergence_Scale` de la série
[Z3-Linq2Z3](../Z3-Linq2Z3/09_Meal_Planner_Convergence_Scale.ipynb). Ce carnet rejoue le
portage **dans son langage d'origine** : API `Microsoft.Z3` brute (Stack B) — la même que le
`09` — et non le DSL Z3.Linq, car le cœur de la leçon est une famille de contraintes
pseudo-booléennes que le DSL n'expose pas.

> **Ce que ce jumeau apporte.** Le corpus réel (cache de [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb))
> est chargé **nativement** (`System.Text.Json`, aucune dépendance Python) et les trois
> encodages sont mesurés **sur le binding .NET**, sans l'overhead `ctypes` par assertion de
> `z3-solver` : les temps publiés ici sont ceux que voit un utilisateur C#. Les **verdicts**
> (A explose, B `unknown`, C `sat`) sont ceux du jumeau Python, remesurés de ce côté.

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< 16c - Capstone patient (Python)](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) | [16e C# >>](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb)

## Objectifs

1. Constater que l'encodage naïf (index + disjonction, celui de [16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb)) **explose dès la construction** à l'échelle réelle.
2. Constater que l'encodage le plus compact à écrire — la **théorie des tableaux** (`Store`/`Select`) — reste **non résolu dans le budget de 15 s** (`unknown`).
3. Maîtriser l'encodage qui passe à l'échelle : le **one-hot pseudo-booléen** (`MkPBEq` / `MkPBLe` / `MkPBGe`).
4. Comprendre **pourquoi** : les recettes sont des contraintes *enablantes*, et la somme pondérée de booléens est du ressort du **solveur pseudo-booléen** natif de Z3, pas de l'arithmétique linéaire générale.

**Prérequis :** [16](Z3-16-Meal-Planner-Python.ipynb) (modélisation, corpus jouet), [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (cache réel), [16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) (idiom index + linking).

## 1. Données : le cache solveur-usable produit par 16b

Le notebook [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) a construit un cache
`data/meals/mealplan_cache.json` : recettes RecipeML appariées à Ciqual, agrégation
**pondérée par la masse**, sous-ensemble *gaté par qualité* (couverture d'appariement `≥ 80%`).
Chaque recette porte un vecteur de `C=5` constituants à l'échelle de la **recette entière**
(kJ, g) — pas per-100g. On charge ce cache en `System.Text.Json` : c'est le seul corpus qui
rende la question de tractabilité *non triviale*.

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Text.Json;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs).
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python).
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");

// --- Chargement du cache solveur-usable produit par 16b (le seul corpus non-jouet).
string MealBase() {
    var cwd = new DirectoryInfo(Directory.GetCurrentDirectory());
    if (cwd.Name == "Z3-API") return Path.Combine(cwd.FullName, "data", "meals");
    for (var d = cwd; d != null; d = d.Parent) {
        var c = Path.Combine(d.FullName, "MyIA.AI.Notebooks", "SymbolicAI", "SMT", "Z3-API", "data", "meals");
        if (Directory.Exists(c)) return c;
    }
    throw new DirectoryNotFoundException("Serie Z3-API introuvable depuis " + cwd.FullName);
}
var cachePath = Path.Combine(MealBase(), "mealplan_cache.json");
if (!File.Exists(cachePath))
    throw new FileNotFoundException($"Cache absent : {cachePath}. Executez d'abord le notebook 16b (couche de donnees) de bout en bout.");

var constituants = new List<string>();
var plats = new List<(string Title, double[] Vec, string[] Cats)>();
int nTotal;
using (var doc = JsonDocument.Parse(File.ReadAllText(cachePath))) {
    nTotal = doc.RootElement.GetProperty("n_total").GetInt32();
    foreach (var e in doc.RootElement.GetProperty("constituants").EnumerateArray())
        constituants.Add(e.GetString()!);
    foreach (var r in doc.RootElement.GetProperty("recipes").EnumerateArray()) {
        var t = (r.GetProperty("title").GetString() ?? "").Trim();
        plats.Add((t[..Math.Min(40, t.Length)],
                   r.GetProperty("vec").EnumerateArray().Select(v => v.GetDouble()).ToArray(),
                   r.GetProperty("cats").EnumerateArray().Select(c => c.GetString()!).ToArray()));
    }
}
int C = constituants.Count;
int R = plats.Count;   // mesure effective au chargement -- jamais recopiee d'une prose ancienne
Console.WriteLine($"Cache charge : R={R} recettes solveur-usables (sur {nTotal} brutes), C={C} constituants.");

// --- contrat assertif : le cache respecte le schema attendu par les notebooks C# de la serie ---
if (C != 5) throw new InvalidOperationException($"C={C} (attendu 5 : Energie, Proteines, Glucides, Lipides, Sel)");
if (plats.Any(p => p.Vec.Length != C)) throw new InvalidOperationException("chaque vec doit avoir C=5 constituants");
if (nTotal < R) throw new InvalidOperationException($"n_total={nTotal} < R={R}");

double Pq(int c, double q) {   // quantile TRONQUE (index dans la liste triee), pas interpole
    var vals = plats.Select(p => p.Vec[c]).OrderBy(v => v).ToList();
    return vals[(int)(q * (vals.Count - 1))];
}

Console.WriteLine("Quartiles par constituant (echelle : recette ENTIERE, agregation ponderee par la masse) :");
for (int c = 0; c < C; c++) {
    var nom = constituants[c][..Math.Min(38, constituants[c].Length)];
    Console.WriteLine($"   [{c}] {nom,-38} P25={Pq(c, 0.25),8:F1}  mediane={Pq(c, 0.5),8:F1}  P75={Pq(c, 0.75),8:F1}");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Moteur : Microsoft.Z3 Z3 4.12.2.0


Cache charge : R=712 recettes solveur-usables (sur 1465 brutes), C=5 constituants.


Quartiles par constituant (echelle : recette ENTIERE, agregation ponderee par la masse) :


   [0] Energie, Règlement UE N° 1169/2011 (kJ P25=  5052,1  mediane=  9995,7  P75= 16417,6


   [1] Protéines, N x facteur de Jones (g/100 P25=    20,6  mediane=    47,8  P75=    76,3


   [2] Glucides (g/100 g)                     P25=    54,9  mediane=   196,4  P75=   397,6


   [3] Lipides (g/100 g)                      P25=    38,1  mediane=   119,1  P75=   245,8


   [4] Sel chlorure de sodium (g/100 g)       P25=     0,8  mediane=     2,6  P75=     6,6


Les quartiles calibrent les **fenêtres nutritionnelles** patient : les vecteurs étant à
l'échelle de la *recette entière* (et non per-100g), des constantes héritées d'un autre
encodage n'auraient aucun sens. La bande d'énergie `[P20, P80] × 5 plats` couvre l'IQR ;
les protéines ont un plancher `P30 × 5` ; le sel un plafond contraignant `P70 × 5`.

In [2]:
// Parametres du theoreme, partages par les trois encodages.
int NMENUS = 7, NPLATS = 5;
// valeurs entieres par constituant (Math.Round = banker's rounding ties-to-even, comme round() Python)
int[][] vint = Enumerable.Range(0, C)
    .Select(c => plats.Select(p => (int)Math.Round(p.Vec[c])).ToArray()).ToArray();
// restrictions patient CALIBREES sur les quartiles mesures ci-dessus.
int loE = NPLATS * (int)Pq(0, 0.20);   // energie/menu : bande large autour de l'IQR
int hiE = NPLATS * (int)Pq(0, 0.80);
int loP = NPLATS * (int)Pq(1, 0.30);   // proteines/menu : plancher realiste
int hiS = Math.Max(1, NPLATS * (int)Pq(4, 0.70));   // sel/menu : plafond contraignant
// CONVENTION : restr = (constituantIndex, lo, hi) avec -1 = "pas de borne sur ce cote".
var restr = new (int cc, int lo, int hi)[] { (0, loE, hiE), (1, loP, -1), (4, -1, hiS) };
Console.WriteLine($"Theoreme : {NMENUS} menus x {NPLATS} plats, sur R={R} recettes (vecteurs recette entiere).");
Console.WriteLine($"   energie/menu in [{loE},{hiE}] kJ, proteines/menu >= {loP} g, sel/menu <= {hiS} g.");

Theoreme : 7 menus x 5 plats, sur R=712 recettes (vecteurs recette entiere).


   energie/menu in [20515,91485] kJ, proteines/menu >= 135 g, sel/menu <= 25 g.


## 2. Trois encodages du même théorème, trois comportements

Le théorème est **fixe** : `7 menus × 5 plats`, chaque plat **distinct** sur la semaine
(variété), chaque menu dans une **fenêtre nutritionnelle**. Seul l'**encodage** SMT change.
On va voir trois comportements radicalement différents sur le **même** corpus réel.

### 2.1 Encodage naïf (disjonction) — explose dès la construction

L'encodage naïf (celui de [16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb)) introduit
une variable de nutrition par créneau et, pour **chaque recette**, la disjonction
`créneau ≠ r  OU  nutrition == ligne[r]`. Le nombre d'assertions croît en
`menus × plats × R × constituants` : on ne mesure même pas la résolution, juste la **construction**.

In [3]:
// Encodage A -- naif : sonde de temps de CONSTRUCTION (la resolution ne demarre meme pas a l'echelle).
void BuildNaive(int rcap) {
    var s = ctx.MkSolver();
    var pid = new IntExpr[NMENUS, NPLATS];
    for (int m = 0; m < NMENUS; m++)
        for (int p = 0; p < NPLATS; p++)
            pid[m, p] = ctx.MkIntConst($"q_{m}_{p}");   // index de recette par creneau
    long na = 0;
    var sw = System.Diagnostics.Stopwatch.StartNew();
    for (int m = 0; m < NMENUS; m++) {
        for (int p = 0; p < NPLATS; p++) {
            s.Assert(ctx.MkAnd(ctx.MkGe(pid[m, p], ctx.MkInt(0)), ctx.MkLt(pid[m, p], ctx.MkInt(rcap))));
            for (int rr = 0; rr < rcap; rr++) {
                foreach (var (cc, lo, hi) in restr) {
                    var nutr = ctx.MkIntConst($"n_{m}_{p}_{cc}");   // meme nom => meme constante Z3
                    s.Assert(ctx.MkOr(ctx.MkNot(ctx.MkEq(pid[m, p], ctx.MkInt(rr))),
                                      ctx.MkEq(nutr, ctx.MkInt(vint[cc][rr]))));   // linking disjonctif
                    na++;
                }
            }
        }
    }
    sw.Stop();
    Console.WriteLine($"  naif R={rcap,5} : {na,8} disjonctions construites en {sw.Elapsed.TotalSeconds:F2}s (CONSTRUCTION seule, sans resolution)");
}
foreach (var rcap in new[] { 100, 300, Math.Min(R, 1000) })
    BuildNaive(rcap);
Console.WriteLine("  -> le cout de construction croit lineairement en R x menus x plats x contraintes :");
Console.WriteLine("     a l'echelle reelle, le solveur n'a meme pas commence a chercher une solution.");

  naif R=  100 :    10500 disjonctions construites en 0,14s (CONSTRUCTION seule, sans resolution)


  naif R=  300 :    31500 disjonctions construites en 0,38s (CONSTRUCTION seule, sans resolution)


  naif R=  712 :    74760 disjonctions construites en 0,89s (CONSTRUCTION seule, sans resolution)


  -> le cout de construction croit lineairement en R x menus x plats x contraintes :


     a l'echelle reelle, le solveur n'a meme pas commence a chercher une solution.


### 2.2 Théorie des tableaux — compact à écrire, mais non résolu dans le budget de 15 s

La théorie des tableaux de Z3 (`Store`/`Select`, axiomes de McCarthy intégrés) exprime le
mapping recette→nutrition de façon **R-indépendante** : un tableau par constituant, construit
comme une chaîne de `R` `Store` concrets, lu par un **index symbolique** `pid[m][p]`. Compact
à écrire... mais dans le budget de 15 s fixé ci-dessous, Z3 ne tranche pas : il renvoie `unknown`.

In [4]:
// Encodage B -- theorie des tableaux : compact a ecrire, mais Z3 -> unknown dans le budget de 15 s (longues chaines de Store).
string Fmt(Status st) => st == Status.SATISFIABLE ? "sat" : st == Status.UNSATISFIABLE ? "unsat" : "unknown";
var sb = ctx.MkSolver();
sb.Set("timeout", 15000u);   // plafond 15s : on attend unknown
var arrs = new List<(int cc, int lo, int hi, ArrayExpr a)>();
foreach (var (cc, lo, hi) in restr) {
    ArrayExpr a = ctx.MkConstArray(ctx.MkIntSort(), ctx.MkInt(0));   // tableau constant 0
    for (int r = 0; r < R; r++)                                      // chaine de R Store concrets
        a = ctx.MkStore(a, ctx.MkInt(r), ctx.MkInt(vint[cc][r]));
    arrs.Add((cc, lo, hi, a));
}
var pid2 = new IntExpr[NMENUS, NPLATS];
var flat = new List<ArithExpr>();
for (int m = 0; m < NMENUS; m++)
    for (int p = 0; p < NPLATS; p++) {
        pid2[m, p] = ctx.MkIntConst($"a_{m}_{p}");
        flat.Add(pid2[m, p]);
    }
foreach (var v in flat) {
    sb.Assert(ctx.MkGe(v, ctx.MkInt(0)));
    sb.Assert(ctx.MkLt(v, ctx.MkInt(R)));
}
sb.Assert(ctx.MkDistinct(flat.Cast<Expr>().ToArray()));   // variete : indices distincts
for (int m = 0; m < NMENUS; m++)
    foreach (var (cc, lo, hi, a) in arrs) {
        var tot = ctx.MkAdd(Enumerable.Range(0, NPLATS)
            .Select(p => (ArithExpr)ctx.MkSelect(a, pid2[m, p])).ToArray());
        if (lo >= 0) sb.Assert(ctx.MkGe(tot, ctx.MkInt(lo)));
        if (hi >= 0) sb.Assert(ctx.MkLe(tot, ctx.MkInt(hi)));
    }
var swB = System.Diagnostics.Stopwatch.StartNew();
var resB = sb.Check();
swB.Stop();
Console.WriteLine($"theorie des tableaux ({NMENUS * NPLATS} index, chaines de Store de longueur {R}) : {Fmt(resB)} en {swB.Elapsed.TotalSeconds:F1}s");
if (resB == Status.UNKNOWN) Console.WriteLine($"   (reason_unknown: {sb.ReasonUnknown})");
Console.WriteLine(resB == Status.UNKNOWN
    ? "  -> compact a ECRIRE, mais non resolu dans le budget de 15 s : Z3 n'a pas tranche. Compacite a ecrire != vitesse de resolution."
    : $"  -> compact a ECRIRE, et tranche ({Fmt(resB)}) dans le budget de 15 s.");

theorie des tableaux (35 index, chaines de Store de longueur 712) : unknown en 15,1s


   (reason_unknown: timeout)


  -> compact a ECRIRE, mais non resolu dans le budget de 15 s : Z3 n'a pas tranche. Compacite a ecrire != vitesse de resolution.


### 2.3 One-hot pseudo-booléen — l'encodage qui passe à l'échelle

L'encodage **one-hot** introduit un booléen `sel[m][p][r]` (« la recette `r` occupe le
créneau `p` du menu `m` »). Trois familles de contraintes, toutes **pseudo-booléennes
natives** de Z3 :

- `MkPBEq(., 1)` par créneau : **exactement une** recette par créneau.
- `MkPBLe(., 1)` par recette sur toute la semaine : chaque recette **au plus une fois** (= variété).
- `MkPBGe` / `MkPBLe` **pondérés** par menu et par constituant : la fenêtre nutritionnelle devient
  une **somme pondérée de booléens** (coefficient = valeur de la recette), traitée par le
  **solveur pseudo-booléen** de Z3 — pas par l'arithmétique linéaire générale.

C'est cet encodage qui passe à l'échelle : les recettes sont des **contraintes *enablantes***,
le solveur trouve un modèle sans énumérer.

> **Idiome .NET.** `MkPBEq(int[] coeffs, BoolExpr[] args, uint k)` prend **deux tableaux
> parallèles** (coefficients, booléens) et le seuil. Côté `z3-solver` Python, `PbEq` fusionne
> les deux en **une liste de paires `(booléen, coefficient)`** — même contrainte, deux
> signatures ; c'est la seule différence d'écriture entre les deux jumeaux sur cette section.

In [5]:
// Encodage C -- one-hot pseudo-booleen : exactly-one + variete + bandes ponderees.
var s = ctx.MkSolver();
var swC = System.Diagnostics.Stopwatch.StartNew();
// sel[m][p][r] : la recette r occupe le creneau p du menu m.
var sel = new BoolExpr[NMENUS][][];
for (int m = 0; m < NMENUS; m++) {
    sel[m] = new BoolExpr[NPLATS][];
    for (int p = 0; p < NPLATS; p++)
        sel[m][p] = Enumerable.Range(0, R).Select(r => ctx.MkBoolConst($"s_{m}_{p}_{r}")).ToArray();
}
var onesR = Enumerable.Repeat(1, R).ToArray();
for (int m = 0; m < NMENUS; m++)                       // exactement 1 recette / creneau  (PbEq, coeffs tous 1, k=1)
    for (int p = 0; p < NPLATS; p++)
        s.Assert(ctx.MkPBEq(onesR, sel[m][p], 1));
var onesWeek = Enumerable.Repeat(1, NMENUS * NPLATS).ToArray();
for (int r = 0; r < R; r++) {                          // variete : chaque recette <= 1x / semaine
    var col = (from m in Enumerable.Range(0, NMENUS)
               from p in Enumerable.Range(0, NPLATS)
               select sel[m][p][r]).ToArray();
    s.Assert(ctx.MkPBLe(onesWeek, col, 1));
}
for (int m = 0; m < NMENUS; m++)                       // fenetre nutritionnelle = somme ponderee de booleens (PB native)
    foreach (var (cc, lo, hi) in restr) {
        var bs = new List<BoolExpr>(); var ws = new List<int>();
        for (int p = 0; p < NPLATS; p++)
            for (int r = 0; r < R; r++) { bs.Add(sel[m][p][r]); ws.Add(vint[cc][r]); }
        if (hi >= 0) s.Assert(ctx.MkPBLe(ws.ToArray(), bs.ToArray(), hi));
        if (lo >= 0) s.Assert(ctx.MkPBGe(ws.ToArray(), bs.ToArray(), lo));
    }
var dtB = swC.Elapsed.TotalSeconds;
var swS = System.Diagnostics.Stopwatch.StartNew();
var resC = s.Check();
swS.Stop();
Console.WriteLine($"one-hot R={R} ({NMENUS * NPLATS * R} booleens) : construction {dtB:F1}s, resolution {swS.Elapsed.TotalSeconds:F1}s -> {Fmt(resC)}");
bool IsTrue(Model m, BoolExpr b) => ((BoolExpr)m.Evaluate(b, true)).BoolValue == Z3_lbool.Z3_L_TRUE;
if (resC == Status.SATISFIABLE) {
    var mo = s.Model;
    for (int m = 0; m < NMENUS; m++) {
        var names = new List<string>();
        for (int p = 0; p < NPLATS; p++)
            for (int r = 0; r < R; r++)
                if (IsTrue(mo, sel[m][p][r])) names.Add(plats[r].Title);
        Console.WriteLine($"  Menu {m + 1} : " + string.Join("  |  ", names.Select(n => n[..Math.Min(18, n.Length)])));
    }
}

one-hot R=712 (24920 booleens) : construction 0,5s, resolution 1,8s -> sat


  Menu 1 : Alligator Sauce Pi  |  Almond Pinecones  |  1-2-3 Cookies  |  Almond Fortune Coo  |  1-2-3-4 Cake with 


  Menu 2 : Alligator Etouffee  |  10-Minute Lasagna  |  14 Karat Cake W/cr  |  1-2-3-4 Cake By Ja  |  1-2-3-4 Cake


  Menu 3 : 3-Bean Bake  |  100% Whole Wheat B  |  1-2-3-4-5 Cake  |  1-1-1 Cookies  |  (Sort of Light) Bo


  Menu 4 : 8 in One Sugar Coo  |  (Sort-Of) Sweet an  |  (Homemade Fresh) C  |  ( From Bread Mix )  |  Aioli Sauce


  Menu 5 : 3-Bean Casserole  |  African Squash and  |  'sense and Sensibi  |  'gimme Both' Pumpk  |  $100 Chocolate Cak


  Menu 6 : 19-Alarm Chili  |  Adas Careh (Lentil  |  1,2,3,4 Cake  |  1,000 Calorie-A-Bi  |  #1 Lemon Bars


  Menu 7 : 16th-Street Stew  |  #10 Cake  |  Almost Dreamland B  |  Acorn Squash Soup  |  Almost Fat-Free Gi


### Bilan des trois encodages

| Encodage | Taille | Comportement à l'échelle réelle |
|---|---|---|
| **Naïf (disjonction)** | `menus × plats × R × C` assertions | **explose à la construction** |
| **Théorie des tableaux** | compact (R-indépendant) | **`unknown`** — non résolu dans le budget de 15 s |
| **One-hot pseudo-booléen** | `menus × plats × R` booléens | **construction + résolu** |

> **Nuance (jusqu'au choix de la contrainte).** Même **au sein** du one-hot, le détail compte :
> exprimer la fenêtre nutritionnelle comme une **contrainte pseudo-booléenne native**
> (`MkPBGe` / `MkPBLe`, somme pondérée de booléens, traitée par le solveur PB) plutôt que comme
> une **somme d'`If`-then-else** routée vers l'arithmétique linéaire générale change la résolution
> d'un facteur **~150×** *(mesuré lors d'une itération antérieure du port C#, à ~1 000 recettes —
> mesure historique non reproduite ici ; le bon outil SMT pour une somme pondérée de booléens est
> le solveur pseudo-booléen, pas le solveur LIA)*.

> **Honnêteté de port.** Ce jumeau mesure ses propres temps sur le binding .NET ; le jumeau
> Python mesure les siens derrière l'overhead `ctypes` de `z3-solver`. Les **valeurs absolues
> diffèrent donc entre les deux carnets**, et c'est attendu : ce qui est comparable — et
> reproduit ici — est le **comportement des trois encodages** (A explose à la construction ;
> B `unknown` ; C `sat`), qui est l'enseignement porté.

## 3. Partitionner par catégorie : un pool de recettes par créneau

Le one-hot plat ci-dessus **peut empiler cinq desserts** dans un même menu (il n'impose aucune
structure de *cours*). En partitionnant les recettes en `5` pools (`Entrée`, `Plat principal`,
`Accompagnement`, `Pain`, `Dessert`) — un pool par créneau — on **réduit `R` par créneau** et
la variété devient triviale (une recette ne peut apparaître qu'à un seul créneau).

In [6]:
// Encodage C' -- partitionnement par categorie : un pool de recettes par creneau.
var COURSES = new[] { "Entree", "Plat principal", "Accompagnement", "Pain", "Dessert" };
var COURSE_CATS = new HashSet<string>[] {
    new HashSet<string> { "appetizers", "soups", "salads", "salad", "soup" },
    new HashSet<string> { "main dish", "meats", "beef", "poultry", "seafood", "fish", "pasta", "casseroles", "chili", "pork", "chicken", "stews" },
    new HashSet<string> { "vegetables", "vegetarian", "sauces", "sauce", "side dishes", "rice", "potatoes" },
    new HashSet<string> { "breads", "bread", "muffins", "rolls", "biscuits" },
    new HashSet<string> { "desserts", "cakes", "cake", "cookies", "chocolate", "fruits", "pies", "candy", "pastries" },
};
int CourseOf(string[] cats) {   // 1er <cat> reconnu gagne ; defaut = plat principal (index 1)
    foreach (var cat in cats)
        for (int k = 0; k < 5; k++) if (COURSE_CATS[k].Contains(cat.ToLowerInvariant())) return k;
    return 1;
}
var pool = Enumerable.Range(0, 5).Select(_ => new List<int>()).ToList();
for (int r = 0; r < R; r++) pool[CourseOf(plats[r].Cats)].Add(r);   // chaque recette -> exactement un pool
Console.WriteLine("Pools par creneau : "
    + string.Join(", ", Enumerable.Range(0, 5).Select(k => $"{COURSES[k]}={pool[k].Count}")));
if (pool.Sum(x => x.Count) != R) throw new InvalidOperationException("les pools doivent partitionner R");
var s3 = ctx.MkSolver();
var sw3 = System.Diagnostics.Stopwatch.StartNew();
// sel3[m][p][j] : le creneau p du menu m prend la j-eme recette DE pool[p]
var sel3 = new BoolExpr[NMENUS][][];
for (int m = 0; m < NMENUS; m++) {
    sel3[m] = new BoolExpr[NPLATS][];
    for (int p = 0; p < NPLATS; p++)
        sel3[m][p] = Enumerable.Range(0, pool[p].Count).Select(j => ctx.MkBoolConst($"c_{m}_{p}_{j}")).ToArray();
}
for (int m = 0; m < NMENUS; m++)   // exactement 1 recette / creneau
    for (int p = 0; p < NPLATS; p++)
        s3.Assert(ctx.MkPBEq(Enumerable.Repeat(1, sel3[m][p].Length).ToArray(), sel3[m][p], 1));
for (int p = 0; p < NPLATS; p++)   // variete : chaque recette <= 1x / semaine (colonne sur les menus)
    for (int j = 0; j < pool[p].Count; j++)
        s3.Assert(ctx.MkPBLe(Enumerable.Repeat(1, NMENUS).ToArray(),
                             Enumerable.Range(0, NMENUS).Select(m => sel3[m][p][j]).ToArray(), 1));
for (int m = 0; m < NMENUS; m++)   // memes bandes PB ponderees
    foreach (var (cc, lo, hi) in restr) {
        var bs = new List<BoolExpr>(); var ws = new List<int>();
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < pool[p].Count; j++) { bs.Add(sel3[m][p][j]); ws.Add(vint[cc][pool[p][j]]); }
        if (hi >= 0) s3.Assert(ctx.MkPBLe(ws.ToArray(), bs.ToArray(), hi));
        if (lo >= 0) s3.Assert(ctx.MkPBGe(ws.ToArray(), bs.ToArray(), lo));
    }
var dtB3 = sw3.Elapsed.TotalSeconds;
int total3 = NMENUS * pool.Sum(x => x.Count);   // = NMENUS x R (partition)
var sw3c = System.Diagnostics.Stopwatch.StartNew();
var res3 = s3.Check();
sw3c.Stop();
Console.WriteLine($"course-onehot : {total3} booleens (contre {NMENUS * NPLATS * R} en one-hot plat) -- construction {dtB3:F1}s, resolution {sw3c.Elapsed.TotalSeconds:F1}s -> {Fmt(res3)}");
if (res3 == Status.SATISFIABLE) {
    var mo3 = s3.Model;
    for (int m = 0; m < 3; m++) {
        var names = new List<string>();
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < pool[p].Count; j++)
                if (IsTrue(mo3, sel3[m][p][j])) names.Add($"{COURSES[p]} : {plats[pool[p][j]].Title}");
        Console.WriteLine($"  Menu {m + 1} -> " + string.Join("  |  ", names.Select(n => n[..Math.Min(30, n.Length)])));
    }
}

Pools par creneau : Entree=52, Plat principal=370, Accompagnement=47, Pain=65, Dessert=178


course-onehot : 4984 booleens (contre 24920 en one-hot plat) -- construction 0,1s, resolution 0,4s -> sat


  Menu 1 -> Entree : Almond Mushroom Pate  |  Plat principal : Almond Quiche  |  Accompagnement : Almond Milk (  |  Pain : Almond Poppy Seed Muffi  |  Dessert : 1993 1st Place Ginge


  Menu 2 -> Entree : Abondigas Venezolanas  |  Plat principal : 19-Alarm Chil  |  Accompagnement : Aaparagus wit  |  Pain : 100% Whole Wheat  |  Dessert : 1,000 Calorie-A-Bite


  Menu 3 -> Entree : Acorn Squash with Cra  |  Plat principal : Alamo Chili  |  Accompagnement : Almond Rice P  |  Pain : Almond Poppyseed Muffin  |  Dessert : Almost Fat-Free Choc


## 4. Restriction patient : le menu végétarien

Même encodage que §3 (pools + one-hot PB), avec une contrainte patient en sus : **interdire**
toute recette viande/poisson. En one-hot, « interdire » = nier le booléen (`MkNot(sel[m][p][j])`).
C'est l'avantage de l'encodage déclaratif : la restriction est une ligne, pas une re-construction.

In [7]:
// Encodage C'' -- restriction patient : menu vegetarien (exclusion categorielle).
var BANNED_VEG = new HashSet<string> { "meats", "beef", "poultry", "seafood", "fish", "pork", "chicken", "stews" };
bool IsMeat(string[] cats) => cats.Any(c => BANNED_VEG.Contains(c.ToLowerInvariant()));
int nForbidden = Enumerable.Range(0, R).Count(r => IsMeat(plats[r].Cats));
var s5 = ctx.MkSolver();
var sel5 = new BoolExpr[NMENUS][][];
for (int m = 0; m < NMENUS; m++) {
    sel5[m] = new BoolExpr[NPLATS][];
    for (int p = 0; p < NPLATS; p++)
        sel5[m][p] = Enumerable.Range(0, pool[p].Count).Select(j => ctx.MkBoolConst($"v_{m}_{p}_{j}")).ToArray();
}
for (int m = 0; m < NMENUS; m++)
    for (int p = 0; p < NPLATS; p++)
        s5.Assert(ctx.MkPBEq(Enumerable.Repeat(1, sel5[m][p].Length).ToArray(), sel5[m][p], 1));
for (int p = 0; p < NPLATS; p++)
    for (int j = 0; j < pool[p].Count; j++)
        s5.Assert(ctx.MkPBLe(Enumerable.Repeat(1, NMENUS).ToArray(),
                             Enumerable.Range(0, NMENUS).Select(m => sel5[m][p][j]).ToArray(), 1));
for (int m = 0; m < NMENUS; m++)
    foreach (var (cc, lo, hi) in restr) {
        var bs = new List<BoolExpr>(); var ws = new List<int>();
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < pool[p].Count; j++) { bs.Add(sel5[m][p][j]); ws.Add(vint[cc][pool[p][j]]); }
        if (hi >= 0) s5.Assert(ctx.MkPBLe(ws.ToArray(), bs.ToArray(), hi));
        if (lo >= 0) s5.Assert(ctx.MkPBGe(ws.ToArray(), bs.ToArray(), lo));
    }
// LA restriction : interdire toute recette viande/poisson dans chaque creneau ou elle pourrait apparaitre.
for (int m = 0; m < NMENUS; m++)
    for (int p = 0; p < NPLATS; p++)
        for (int j = 0; j < pool[p].Count; j++)
            if (IsMeat(plats[pool[p][j]].Cats)) s5.Assert(ctx.MkNot(sel5[m][p][j]));
var sw5 = System.Diagnostics.Stopwatch.StartNew();
var res5 = s5.Check();
sw5.Stop();
Console.WriteLine($"vegetarien : {nForbidden} recettes viande/poisson interdites -- resolution {sw5.Elapsed.TotalSeconds:F1}s -> {Fmt(res5)}");
if (res5 == Status.SATISFIABLE) {
    var mo5 = s5.Model;
    var names = new List<string>();
    for (int p = 0; p < NPLATS; p++)
        for (int j = 0; j < pool[p].Count; j++)
            if (IsTrue(mo5, sel5[0][p][j])) names.Add($"{COURSES[p]} : {plats[pool[p][j]].Title}");
    Console.WriteLine("  Menu vegetarien 1 -> " + string.Join("  |  ", names.Select(n => n[..Math.Min(30, n.Length)])));
}

vegetarien : 46 recettes viande/poisson interdites -- resolution 0,4s -> sat


  Menu vegetarien 1 -> Entree : Abalone Soup #2  |  Plat principal : Alan Thicke's  |  Accompagnement : African Veget  |  Pain : 100% Whole Wheat Bread   |  Dessert : 1-2-3-4-5 Cake


## 5. Exercices

Quatre exercices prolongent l'étude des encodages. Chacun manipule l'encodage **C** (one-hot
pseudo-booléen) ou ses variantes `C'` / `C''`. Les variables `s`, `sel`, `vint`, `restr`,
`plats`, `pool` restent en portée.

### Exercice 1 — Serrer le plafond d'énergie jusqu'à `UNSAT`

On a calibré `hiE` sur `P80(énergie)`. Que se passe-t-il si on **resserre** ce plafond
(vers `P20`, puis `P10`) ? L'espace des modèles se vide : à partir d'un seuil, le théorème
devient **insatisfiable**. Construisez l'encodage C avec un `hiE` paramétrique et
tracez la frontière `SAT` → `UNSAT`.

In [8]:
// Exercice 1 -- serrer hiE jusqu'a UNSAT.
// TODO: boucler sur des hiE decroissants (ex: P80, P50, P30, P20, P10), reconstruire l'encodage C,
//       enregistrer le verdict (sat/unsat) et le temps, tracer la frontiere.
Console.WriteLine("Exercice 1 a completer : frontiere SAT -> UNSAT quand hiE diminue.");

Exercice 1 a completer : frontiere SAT -> UNSAT quand hiE diminue.


### Exercice 2 — Plancher de glucides

Le constituant glucides (index `2`) est **chargé mais non contraint** dans `restr`. Ajoutez
un plancher `MkPBGe(pairs, GLUCIDES_MIN)` sur les glucides par menu (calibre sur `P30` des
glucides).

In [9]:
// Exercice 2 -- plancher de glucides (constituant index 2, non contraint dans restr).
// TODO: definir GLUCIDES_MIN = NPLATS * (int)Pq(2, 0.30), puis ajouter a l'encodage C la contrainte
//       ctx.MkPBGe(coeffs, [(sel[m][p][r] pour p, r)], GLUCIDES_MIN) par menu.
Console.WriteLine("Exercice 2 a completer : plancher de glucides par menu via MkPBGe.");

Exercice 2 a completer : plancher de glucides par menu via MkPBGe.


### Exercice 3 — Exclusion d'allergène (mot-clé)

Sur le modèle de l'encodage C'' (végétarien), interdisez les recettes dont un ingrédient
contient un mot-clé d'allergène (ex : `"gluten"`, `"dairy"`, `"peanut"`). En one-hot, cela se
ramène à `MkNot(sel[m][p][j])` pour chaque recette touchée.

In [10]:
// Exercice 3 -- exclusion d'allergene (mot-cle sur les cats / titre).
// TODO: definir ALLERGENS = {"gluten", "dairy", "peanut"}, identifier les recettes touchees
//       (plats[r].Cats ou plats[r].Title), ajouter MkNot(sel[]) comme en C''.
Console.WriteLine("Exercice 3 a completer : exclusion d'allergenes via MkNot(sel).");

Exercice 3 a completer : exclusion d'allergenes via MkNot(sel).


### Exercice 4 — Mesurer l'explosion

L'encodage A « explose à la construction » ; le C « passe à l'échelle ». **Mesurez-le** :
balayez `R` sur `{100, 300, R}` pour les deux encodages et tracez le coût de construction
(naïf) vs résolution (one-hot) en fonction de `R`.

In [11]:
// Exercice 4 -- mesurer l'explosion (sweep R pour A et C).
// TODO: pour rcap dans (100, 300, R): mesurer BuildNaive(rcap) [construction] et un build+check
//       one-hot sur rcap premieres recettes ; tracer R vs temps pour les deux encodages.
Console.WriteLine("Exercice 4 a completer : sweep R, tracer construction (naif) vs resolution (one-hot).");

Exercice 4 a completer : sweep R, tracer construction (naif) vs resolution (one-hot).


## Synthèse — l'encodage décide de la tractabilité

Le même théorème (`7 menus × 5 plats`, variété, fenêtre nutritionnelle), sur le **même**
corpus réel, donne trois comportements radicalement différents selon l'encodage SMT :

- l'**index + disjonction** naïf explose dès la **construction** (`O(menus × plats × R × C)`) ;
- la **théorie des tableaux**, si compacte à écrire, reste **`unknown`** ;
- le **one-hot pseudo-booléen** se construit et se résout, parce que la somme pondérée de
  booléens est traitée par le **solveur pseudo-booléen natif** de Z3.

**Leçon :** à l'échelle, ce n'est pas la puissance brute du solveur qui décide — c'est le
choix d'**encodage**. Les recettes sont des contraintes *enablantes* ; l'encodage décide si
l'espace qu'elles ouvrent est explorable. C'est la clôture du bloc meal-planner côté C# : de
la modélisation jouet ([16](Z3-16-Meal-Planner-Python.ipynb)) à la donnée réelle
([16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb)), au capstone patient
([16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb)), jusqu'à la tractabilité à
l'échelle (ce carnet), avant l'optimisation de [16e C#](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb).